In [18]:
from pinecone import Pinecone, ServerlessSpec
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")

pc = Pinecone(
    api_key=api_key,
)

dense_index_name = "rag-actuaria-2"
if not pc.has_index(dense_index_name):
    pc.create_index(
        name=dense_index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )

dense_index = pc.Index(dense_index_name)
dense_index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 258}},
 'total_vector_count': 258,
 'vector_type': 'dense'}

In [19]:
from chonkie import MarkdownChef
import os
import re
import yaml
from pprint import pprint
from chonkie import LateChunker

folders = [
    "./sample_docs/global_docs",
    # Add more directories if needed
]

all_docs = []

total_files = 0
for folder in folders:
    for filename in os.listdir(folder):
        if filename.endswith(".md"):
            total_files += 1
            md_path = os.path.join(folder, filename)
            with open(md_path, "r", encoding="utf-8") as f:
                content = f.read()
                match = re.search(r"^---\s*([\s\S]+?)---\s*", content, re.MULTILINE)
                if match:
                    metadata_block = match.group(1)
                    metadata = yaml.safe_load(metadata_block)
                    metadata_str = match.group(0).strip()
                    metadata_lines = [line.strip() for line in metadata_str.splitlines() if line.strip() and not line.strip().startswith('---')]
                else:
                    metadata = {}
                    metadata_lines = []
            doc = MarkdownChef().process(path=md_path)
            doc.filename = filename  # Tambahkan filename ke MarkdownDocument
            doc.metadata = metadata  # Tambahkan metadata ke MarkdownDocument
            # for table in getattr(doc, 'tables', []):
            #     table.filename = filename
            # for code in getattr(doc, 'code', []):
            #     code.filename = filename
            # for image in getattr(doc, 'images', []):
            #     image.filename = filename
            for chunk in doc.chunks:
                # chunk.filename = filename
                # chunk.metadata = metadata
                if metadata_lines:
                    chunk_lines = chunk.text.splitlines()
                    cleaned_lines = [line for line in chunk_lines if not any(line.strip() == mline for mline in metadata_lines)]
                    chunk.text = '\n'.join(cleaned_lines).strip()
            all_docs.append(doc)

print(f"Jumlah dokumen Markdown yang diproses: {total_files}")


2025-11-10 22:45:00.542 | DEBUG    | chonkie.chef.base:read:68 - Reading file: ./sample_docs/global_docs/assumptions_reference.md
2025-11-10 22:45:00.543 | DEBUG    | chonkie.chef.base:read:71 - Successfully read file: ./sample_docs/global_docs/assumptions_reference.md
2025-11-10 22:45:00.543 | DEBUG    | chonkie.chef.markdown:parse:200 - Processing markdown text: 9142 characters
2025-11-10 22:45:00.543 | INFO     | chonkie.chef.markdown:parse:210 - Markdown processing complete: extracted 1 tables, 4 code blocks, 0 images, 6 chunks
2025-11-10 22:45:00.545 | DEBUG    | chonkie.chef.base:read:68 - Reading file: ./sample_docs/global_docs/INDEX.md
2025-11-10 22:45:00.546 | DEBUG    | chonkie.chef.base:read:71 - Successfully read file: ./sample_docs/global_docs/INDEX.md
2025-11-10 22:45:00.546 | DEBUG    | chonkie.chef.markdown:parse:200 - Processing markdown text: 20827 characters
2025-11-10 22:45:00.547 | INFO     | chonkie.chef.markdown:parse:210 - Markdown processing complete: extracted

Jumlah dokumen Markdown yang diproses: 34


In [20]:
for doc in all_docs:
    # Access the extracted components
    # print(f"Found {len(doc.tables)} tables")
    # print(f"Found {len(doc.code)} code blocks")
    # print(f"Found {len(doc.images)} images")
    print(f"Found {len(doc.chunks)} text chunks")

Found 6 text chunks
Found 16 text chunks
Found 5 text chunks
Found 23 text chunks
Found 2 text chunks
Found 13 text chunks
Found 18 text chunks
Found 3 text chunks
Found 12 text chunks
Found 4 text chunks
Found 7 text chunks
Found 3 text chunks
Found 35 text chunks
Found 0 text chunks
Found 33 text chunks
Found 12 text chunks
Found 9 text chunks
Found 13 text chunks
Found 2 text chunks
Found 26 text chunks
Found 9 text chunks
Found 22 text chunks
Found 48 text chunks
Found 9 text chunks
Found 3 text chunks
Found 38 text chunks
Found 8 text chunks
Found 1 text chunks
Found 9 text chunks
Found 11 text chunks
Found 25 text chunks
Found 8 text chunks
Found 6 text chunks
Found 17 text chunks


simpan ke dlm json

In [21]:
import json
import datetime

def dataclass_to_dict(obj):
    # Recursively convert dataclass (and nested dataclasses) to dict, including filename
    if hasattr(obj, "__dict__"):
        result = {}
        for k, v in obj.__dict__.items():
            if isinstance(v, list):
                result[k] = [dataclass_to_dict(i) for i in v]
            else:
                result[k] = dataclass_to_dict(v)
        return result
    elif isinstance(obj, dict):
        return {k: dataclass_to_dict(v) for k, v in obj.items()}
    elif isinstance(obj, (datetime.date, datetime.datetime)):
        return obj.isoformat()
    else:
        return obj

# Konversi semua doc ke dict
all_dicts = [dataclass_to_dict(doc) for doc in all_docs]

# Export ke file JSON
with open("exported_doc.json", "w", encoding="utf-8") as f:
    json.dump(all_dicts, f, ensure_ascii=False, indent=2)


upsert via chonkie handshake

upsert via langchain pinecone

In [22]:
from uuid import uuid4
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Use the existing dense_index (Pinecone Index object)
vector_store = PineconeVectorStore(index=dense_index, embedding=embeddings)

In [23]:
import json
from pprint import pprint
from langchain_core.documents import Document
from uuid import uuid4

with open("exported_doc.json", "r", encoding="utf-8") as f:
    docs = json.load(f)

MAX_METADATA_SIZE = 40960  # Turunkan buffer untuk safety (40KB instead of 40KB)

def split_large_metadata(metadata, max_size=MAX_METADATA_SIZE):
    """
    Split metadata yang terlalu besar dengan memotong field text yang panjang
    """
    meta_str = json.dumps(metadata, ensure_ascii=False)
    current_size = len(meta_str.encode('utf-8'))
    
    if current_size <= max_size:
        return metadata
    
    # Buat copy metadata
    reduced_meta = metadata.copy()
    
    # Cari field text yang paling besar
    text_fields = {}
    for key, value in reduced_meta.items():
        if isinstance(value, str):
            text_fields[key] = len(value.encode('utf-8'))
    
    # Sort by size (terbesar dulu)
    sorted_fields = sorted(text_fields.items(), key=lambda x: x[1], reverse=True)
    
    # Potong field text yang besar sampai di bawah limit
    for field_name, field_size in sorted_fields:
        if current_size <= max_size:
            break
        
        # Potong field ini jadi 50% atau hapus
        original_value = reduced_meta[field_name]
        if field_size > 10000:  # Kalau field > 10KB, potong
            # Potong jadi max 5000 chars
            reduced_meta[field_name] = original_value[:5000] + "... [TRUNCATED]"
        else:
            # Kalau kecil, hapus aja
            del reduced_meta[field_name]
        
        # Hitung ulang size
        meta_str = json.dumps(reduced_meta, ensure_ascii=False)
        current_size = len(meta_str.encode('utf-8'))
        print(f"  ✂️ Reduced field '{field_name}' from {field_size} bytes → current total: {current_size} bytes")
    
    return reduced_meta

skipped_chunks = []
processed_chunks = []

for doc in docs:
    chunk_documents = []
    for chunk in doc['chunks']:
        meta = chunk.get('metadata', {})
        meta_size = len(json.dumps(meta, ensure_ascii=False).encode('utf-8'))
        
        # Jika metadata terlalu besar, coba reduce
        if meta_size > MAX_METADATA_SIZE:
            print(f"⚠️ Metadata terlalu besar di file {doc.get('filename', '-')}: {meta_size} bytes")
            print(f"   Keys: {list(meta.keys())}")
            
            # Auto-reduce metadata
            reduced_meta = split_large_metadata(meta, MAX_METADATA_SIZE)
            reduced_size = len(json.dumps(reduced_meta, ensure_ascii=False).encode('utf-8'))
            
            if reduced_size > MAX_METADATA_SIZE:
                # Kalau masih terlalu besar, skip
                skipped_chunks.append({
                    'filename': doc.get('filename', '-'),
                    'original_size': meta_size,
                    'reduced_size': reduced_size,
                    'meta_keys': list(meta.keys())
                })
                print(f"  ⏩ Masih terlalu besar setelah reduce ({reduced_size} bytes), SKIP chunk ini")
                continue
            else:
                print(f"  ✅ Berhasil reduce: {meta_size} → {reduced_size} bytes")
                meta = reduced_meta
                processed_chunks.append({
                    'filename': doc.get('filename', '-'),
                    'original_size': meta_size,
                    'reduced_size': reduced_size
                })
        
        doc_obj = Document(
            page_content=chunk['text'],
            metadata=meta
        )
        chunk_documents.append(doc_obj)
    
    if chunk_documents:
        print(f"\n📄 Document objects for file: {doc.get('filename', '-')}")
        print(f"   Total chunks: {len(chunk_documents)}")
        
        # Add to vector store
        uuids = [str(uuid4()) for _ in chunk_documents]
        vector_store.add_documents(documents=chunk_documents, ids=uuids)
        print(f"   ✅ Added {len(chunk_documents)} chunks to vector store")

# Summary
print("\n" + "="*60)
print(f"📊 SUMMARY:")
print(f"   Chunks yang di-reduce: {len(processed_chunks)}")
print(f"   Chunks yang di-skip: {len(skipped_chunks)}")

if processed_chunks:
    print("\n✂️ Chunks yang di-reduce:")
    for i, info in enumerate(processed_chunks, 1):
        print(f"   [{i}] {info['filename']}: {info['original_size']} → {info['reduced_size']} bytes")

if skipped_chunks:
    print("\n⏩ Chunks yang di-skip (tidak bisa di-reduce):")
    for i, info in enumerate(skipped_chunks, 1):
        print(f"   [{i}] {info['filename']}: {info['original_size']} bytes (after reduce: {info['reduced_size']})")
        print(f"       Keys: {info['meta_keys']}")


📄 Document objects for file: assumptions_reference.md
   Total chunks: 6
   ✅ Added 6 chunks to vector store

📄 Document objects for file: INDEX.md
   Total chunks: 16
   ✅ Added 6 chunks to vector store

📄 Document objects for file: INDEX.md
   Total chunks: 16
   ✅ Added 16 chunks to vector store

📄 Document objects for file: 02f2_valuasi_analisis_lanjutan.md
   Total chunks: 5
   ✅ Added 16 chunks to vector store

📄 Document objects for file: 02f2_valuasi_analisis_lanjutan.md
   Total chunks: 5
   ✅ Added 5 chunks to vector store

📄 Document objects for file: step01_employee_data.md
   Total chunks: 23
   ✅ Added 5 chunks to vector store

📄 Document objects for file: step01_employee_data.md
   Total chunks: 23
   ✅ Added 23 chunks to vector store

📄 Document objects for file: 05d_faq_sistem.md
   Total chunks: 2
   ✅ Added 23 chunks to vector store

📄 Document objects for file: 05d_faq_sistem.md
   Total chunks: 2
   ✅ Added 2 chunks to vector store

📄 Document objects for file: 01

rag

In [ ]:
# results = vector_store.similarity_search(
#     "LangChain provides abstractions to make working with LLMs easy",
#     k=2,
#     filter={"source": "tweet"},
# )
# for res in results:
#     print(f"* {res.page_content} [{res.metadata}]")